# Process Effect Lab: What-if Notebook

この Notebook は、モデルをただ実行するためではなく、**何を変えると何が起きるか**を確認するためのものです。

進め方は単純です。

1. 1 つのパラメータだけ変える。
2. グラフと数値を両方見る。
3. 「なぜそうなったか」を自分の言葉で説明する。

この Notebook の計算本体は `effect_lab_core.py` にあります。NumPy/Pandas ベースで、OLS、Ridge、PLS1、JIT 型局所モデル、グレーボックス、転移学習、データ診断を実装しています。


## 0. セットアップ

まず下のセルを実行してください。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from effect_lab_core import (
    simulate_regression,
    simulate_multicollinearity,
    simulate_jit,
    simulate_graybox,
    simulate_transfer,
    make_raw_signal,
    diagnose_raw_signal,
)

pd.set_option("display.max_columns", 100)


## 0.1 何を変えるとどうなるか

まず全体の見取り図です。細かい式より先に、この表を頭に入れてください。


In [ ]:
knobs = pd.DataFrame([
    ["単回帰", "y_noise", "増やす", "相関・R²・標準化傾きが下がる"],
    ["単回帰", "x_noise", "増やす", "OLS の x は正確という前提が崩れ、傾きがずれやすい"],
    ["単回帰", "outlier_strength", "増やす", "少数点に回帰線が引っ張られる"],
    ["多重共線性", "collinearity", "増やす", "条件数・VIF・係数のばらつきが上がる"],
    ["PLS", "pls_components", "増やす", "情報は増えるが、増やしすぎると OLS に近づく"],
    ["JIT", "k", "増やす", "ノイズには強くなるが、別状態のデータも混ざる"],
    ["JIT", "bandwidth", "増やす", "遠いデータまで効く。大きすぎると全体モデルに近づく"],
    ["グレーボックス", "physics_bias", "増やす", "物理モデル単独の系統誤差が大きくなる"],
    ["グレーボックス", "parameter_drift", "増やす", "Serial 型が効きやすくなる"],
    ["グレーボックス", "residual_observability", "増やす", "Parallel / Combined の統計補正が効きやすくなる"],
    ["転移", "n_target", "増やす", "Target only が強くなり、転移の必要性が下がる"],
    ["転移", "domain_gap", "増やす", "素朴な転移が害になりやすい"],
    ["生データ", "range_shift / trend / cycle", "増やす", "モデル以前に、分割・前処理・運転確認が必要になる"],
], columns=["テーマ", "変えるもの", "変え方", "起きやすいこと"])
knobs


---

# 1. 単回帰：OLS はどの線を引くのか

### ここで理解すること

- OLS は **y 方向の誤差**を小さくします。
- x と y を平等に扱うわけではありません。
- x と y を標準化した単回帰では、傾きは相関係数になります。

### まず触るパラメータ

- `y_noise`: y 側のノイズ。増やすと相関が下がる。
- `x_noise`: x 側の測定誤差。OLS の前提が崩れる。
- `outlier_strength`: 外れ値の強さ。少数点に線が引っ張られる。


In [ ]:
params = dict(
    n=80,
    true_slope=1.0,
    y_noise=0.6,
    x_noise=0.0,
    outlier_strength=0.0,
    seed=0,
)

r = simulate_regression(**params)
df = r["data"]
print(r["metrics"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["x_observed"], df["y"], s=28)
order = np.argsort(df["x_observed"].to_numpy())
ax.plot(df["x_observed"].to_numpy()[order], df["y_pred_ols"].to_numpy()[order], label="OLS")
ax.set_xlabel("x observed")
ax.set_ylabel("y")
ax.legend()
ax.set_title("単回帰の結果")
plt.show()


### 実験 1-A: `y_noise` を増やす

`y_noise` を増やすと、散布図がぼやけます。すると相関が下がり、標準化後の傾きも下がります。


In [ ]:
rows = []
for y_noise in np.linspace(0.05, 2.0, 12):
    rr = simulate_regression(y_noise=float(y_noise), x_noise=0.0, seed=1)
    rows.append({"y_noise": y_noise, **rr["metrics"]})
noise_sweep = pd.DataFrame(rows)
display(noise_sweep[["y_noise", "corr", "standardized_ols_slope", "r2", "rmse"]])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(noise_sweep["y_noise"], noise_sweep["corr"], marker="o", label="corr / standardized slope")
ax.plot(noise_sweep["y_noise"], noise_sweep["r2"], marker="o", label="R2")
ax.set_xlabel("y_noise")
ax.set_ylabel("metric")
ax.legend()
ax.set_title("y_noise を増やしたときの変化")
plt.show()


---

# 2. 多重共線性・PLS：係数の大きさを信じてよいか

### ここで理解すること

- 入力変数同士が似た動きをすると、係数が激しく不安定になります。
- テスト RMSE が良くても、係数解釈は危険なことがあります。
- PLS は潜在変数を使って、多重共線性の悪影響を抑えます。
- ただし潜在変数数を増やしすぎると、OLS に近づきます。


In [ ]:
params = dict(
    n_train=80,
    n_test=400,
    collinearity=0.95,
    y_noise=0.25,
    n_features=8,
    pls_components=2,
    ridge_alpha=1.0,
    seed=0,
)

r = simulate_multicollinearity(**params)
print("summary:", r["summary"])
display(r["metrics"])
display(r["stability"])
display(r["coefficients"])

def plot_coefficients(coef_df):
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(coef_df))
    width = 0.25
    ax.bar(x - width, coef_df["OLS"], width, label="OLS")
    ax.bar(x, coef_df["Ridge"], width, label="Ridge")
    ax.bar(x + width, coef_df["PLS"], width, label="PLS")
    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(coef_df["feature"], rotation=45, ha="right")
    ax.set_ylabel("coefficient")
    ax.legend()
    ax.set_title("係数の比較")
    plt.show()

plot_coefficients(r["coefficients"])


### 実験 2-A: `collinearity` を増やす

条件数と VIF が上がり、係数のばらつきが大きくなる様子を見ます。


In [ ]:
rows = []
for c in np.linspace(0.1, 0.99, 10):
    rr = simulate_multicollinearity(collinearity=float(c), pls_components=2, perturb_repeats=20, seed=2)
    ols_stab = rr["stability"].set_index("model").loc["OLS"]
    pls_stab = rr["stability"].set_index("model").loc["PLS"]
    rows.append({
        "collinearity": c,
        "condition_number": rr["summary"]["condition_number"],
        "max_vif": rr["summary"]["max_vif"],
        "OLS_coef_std_mean": ols_stab["coef_std_mean"],
        "PLS_coef_std_mean": pls_stab["coef_std_mean"],
    })
mc_sweep = pd.DataFrame(rows)
display(mc_sweep)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mc_sweep["collinearity"], mc_sweep["OLS_coef_std_mean"], marker="o", label="OLS")
ax.plot(mc_sweep["collinearity"], mc_sweep["PLS_coef_std_mean"], marker="o", label="PLS")
ax.set_xlabel("collinearity")
ax.set_ylabel("mean coefficient std")
ax.set_title("微小ノイズに対する係数の不安定性")
ax.legend()
plt.show()


### 実験 2-B: PLS の潜在変数数を変える

`pls_components` を増やすほど情報は増えます。ただし、全成分を使うと OLS に近づきます。CV の最小値だけで決めず、係数の安定性も見てください。


In [ ]:
r = simulate_multicollinearity(collinearity=0.97, n_features=8, pls_components=2, seed=4)
display(r["cv"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r["cv"]["components"], r["cv"]["CV_RMSE"], marker="o")
ax.set_xlabel("PLS components")
ax.set_ylabel("CV_RMSE")
ax.set_title("潜在変数数と CV_RMSE")
plt.show()


---

# 3. JIT 型モデル：近いデータだけをどう使うか

### ここで理解すること

- 固定モデルは、状態変化やメンテナンスで外れやすくなります。
- JIT 型モデルは、現在の運転点に近い過去データを使います。
- `k` と `bandwidth` は、局所性と安定性のトレードオフです。


In [ ]:
params = dict(
    x1_current=0.5,
    x2_current=0.0,
    wear_current=0.85,
    nonlinearity=1.3,
    drift_strength=1.0,
    noise=0.12,
    k=35,
    bandwidth=0.8,
    pls_components=2,
    seed=0,
)

r = simulate_jit(**params)
print("true y:", r["y_true"])
display(r["metrics"])

db = r["database"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(db["time"], db["y"], marker="o", markersize=2, linewidth=1)
ax.set_xlabel("time")
ax.set_ylabel("quality y")
ax.set_title("過去データベース")
plt.show()


### 実験 3-A: `k` と `bandwidth` を変える

- `k` が小さい：局所的だがノイズに弱い。
- `k` が大きい：安定するが、違う運転状態が混ざる。
- `bandwidth` が小さい：近い点だけ効く。
- `bandwidth` が大きい：遠い点も効く。


In [ ]:
rows = []
for k in [3, 5, 10, 20, 40, 80, 120]:
    for bw in [0.25, 0.5, 0.8, 1.2, 2.0]:
        rr = simulate_jit(k=k, bandwidth=bw, wear_current=0.85, drift_strength=1.2, seed=5)
        lw = rr["metrics"].set_index("method").loc["LW-PLS"]
        rows.append({"k": k, "bandwidth": bw, "abs_error": lw["abs_error_vs_truth"], "effective_n": lw["effective_n"]})
jit_sweep = pd.DataFrame(rows)
display(jit_sweep.head())

fig, ax = plt.subplots(figsize=(7, 4))
for bw, g in jit_sweep.groupby("bandwidth"):
    ax.plot(g["k"], g["abs_error"], marker="o", label=f"bandwidth={bw}")
ax.set_xlabel("k")
ax.set_ylabel("abs error")
ax.set_title("JIT: k と bandwidth の効果")
ax.legend(fontsize=8)
plt.show()


---

# 4. グレーボックス：物理モデルに何を補正させるか

### ここで理解すること

- 物理モデルには、系統誤差や未知パラメータの変化があります。
- Parallel 型は「物理モデルの残差」を統計モデルで補正します。
- Serial 型は「物理モデル内のパラメータ」を統計モデルで推定します。
- Combined 型は両方を使います。


In [ ]:
params = dict(
    physics_bias=0.3,
    parameter_drift=0.7,
    residual_strength=0.8,
    residual_observability=0.8,
    noise=0.10,
    ridge_alpha=1.0,
    seed=0,
)

r = simulate_graybox(**params)
display(r["metrics"])

best = r["metrics"].iloc[0]["model"]
pred_df = r["test_predictions"]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pred_df["y"], pred_df[best], s=18)
lo = min(pred_df["y"].min(), pred_df[best].min())
hi = max(pred_df["y"].max(), pred_df[best].max())
ax.plot([lo, hi], [lo, hi], linestyle="--")
ax.set_xlabel("measured y")
ax.set_ylabel(f"predicted y: {best}")
ax.set_title("最良モデルの予測 vs 実測")
plt.show()


### 実験 4-A: `residual_observability` を変える

残差が測定変数から説明できるほど、Parallel / Combined の補正が効きます。逆に、残差が観測不能な外乱なら、統計モデルを足しても改善しにくいです。


In [ ]:
rows = []
for obs in np.linspace(0, 1, 11):
    rr = simulate_graybox(residual_observability=float(obs), residual_strength=0.9, parameter_drift=0.7, physics_bias=0.4, seed=3)
    for _, row in rr["metrics"].iterrows():
        rows.append({"residual_observability": obs, **row.to_dict()})
gray_sweep = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4))
for model, g in gray_sweep.groupby("model"):
    ax.plot(g["residual_observability"], g["RMSE_test"], marker="o", label=model)
ax.set_xlabel("residual_observability")
ax.set_ylabel("RMSE_test")
ax.set_title("残差の観測可能性と補正効果")
ax.legend(fontsize=8)
plt.show()


---

# 5. 転移学習：過去データを使うと本当に得か

### ここで理解すること

- ターゲットデータが少ないほど、ソースデータの価値が上がります。
- ソースとターゲットが違いすぎると、素朴な転移は害になります。
- 共通変数と固有変数を分ける特徴空間は、安全側の転移になります。


In [ ]:
params = dict(
    n_source=300,
    n_target=10,
    n_test=300,
    n_common=8,
    n_source_unique=3,
    n_target_unique=3,
    domain_gap=0.35,
    transfer_weight=0.6,
    ridge_alpha=3.0,
    noise=0.25,
    seed=0,
)

r = simulate_transfer(**params)
display(r["metrics"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(r["metrics"]["model"], r["metrics"]["RMSE_target_test"])
ax.tick_params(axis="x", rotation=25)
ax.set_ylabel("RMSE_target_test")
ax.set_title("転移学習の比較")
plt.show()


### 実験 5-A: `n_target` と `domain_gap` を変える

- `n_target` が少ない：転移が効きやすい。
- `domain_gap` が大きい：素朴な転移は危険。


In [ ]:
rows = []
for n_target in [3, 5, 10, 20, 50, 100]:
    for gap in [0.1, 0.35, 0.7, 1.2]:
        rr = simulate_transfer(n_target=n_target, domain_gap=gap, transfer_weight=0.6, seed=6)
        for _, row in rr["metrics"].iterrows():
            rows.append({"n_target": n_target, "domain_gap": gap, **row.to_dict()})
transfer_sweep = pd.DataFrame(rows)
display(transfer_sweep.head())

fig, ax = plt.subplots(figsize=(8, 4))
sub = transfer_sweep[transfer_sweep["domain_gap"] == 0.35]
for model, g in sub.groupby("model"):
    ax.plot(g["n_target"], g["RMSE_target_test"], marker="o", label=model)
ax.set_xlabel("n_target")
ax.set_ylabel("RMSE_target_test")
ax.set_title("ターゲットデータ数の効果")
ax.legend(fontsize=8)
plt.show()


---

# 6. 生データ診断：モデル前に見る

### ここで理解すること

モデルを作る前に、データの形を見ます。外れ値、レンジ変更、トレンド、周期、欠損、下限張り付き、複数モードは、モデル選択より先に確認すべき問題です。


In [ ]:
params = dict(
    n=300,
    outlier_strength=4.0,
    range_shift=1.0,
    trend_strength=0.5,
    cycle_strength=0.8,
    missing_rate=0.05,
    lower_clip=0.0,
    two_modes=0.0,
    seed=0,
)

df = make_raw_signal(**params)
diag = diagnose_raw_signal(df)
display(diag)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["time"], df["value"], marker="o", markersize=2, linewidth=1)
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.set_title("生データの時系列")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["value"].dropna(), bins=30)
ax.set_xlabel("value")
ax.set_ylabel("count")
ax.set_title("ヒストグラム")
plt.show()


### 実験 6-A: 異常の種類を 1 つずつ入れる

下の `cases` を実行して、どの診断フラグが立つか見てください。


In [ ]:
cases = {
    "outlier": dict(outlier_strength=8.0, range_shift=0, trend_strength=0, cycle_strength=0, missing_rate=0, lower_clip=0, two_modes=0),
    "range_shift": dict(outlier_strength=0, range_shift=2.0, trend_strength=0, cycle_strength=0, missing_rate=0, lower_clip=0, two_modes=0),
    "trend": dict(outlier_strength=0, range_shift=0, trend_strength=1.5, cycle_strength=0, missing_rate=0, lower_clip=0, two_modes=0),
    "cycle": dict(outlier_strength=0, range_shift=0, trend_strength=0, cycle_strength=2.0, missing_rate=0, lower_clip=0, two_modes=0),
    "missing": dict(outlier_strength=0, range_shift=0, trend_strength=0, cycle_strength=0, missing_rate=0.2, lower_clip=0, two_modes=0),
    "lower_clip": dict(outlier_strength=0, range_shift=0, trend_strength=0, cycle_strength=0, missing_rate=0, lower_clip=0.2, two_modes=0),
    "two_modes": dict(outlier_strength=0, range_shift=0, trend_strength=0, cycle_strength=0, missing_rate=0, lower_clip=0, two_modes=2.0),
}

rows = []
for name, kwargs in cases.items():
    df_case = make_raw_signal(n=300, seed=1, **kwargs)
    diag_case = diagnose_raw_signal(df_case)
    for _, row in diag_case.iterrows():
        rows.append({"case": name, **row.to_dict()})
case_diag = pd.DataFrame(rows)
display(case_diag[case_diag["flag"] == True])


---

# 7. 自分の言葉で答える確認問題

下に答えを書いてください。答えを文章にできるかどうかが大事です。

1. 多重共線性が強いとき、テスト RMSE が小さければ係数解釈も信頼できるか。
2. PLS の潜在変数数を増やし続けると、なぜ OLS に近づくのか。
3. JIT 型モデルで `k` を小さくしすぎると何が起きるか。
4. グレーボックスで、統計補正が効かない残差とはどんな残差か。
5. 転移学習で、ソースデータをたくさん使うほど常に良くなるか。
6. 生データに周期変動がある場合、普通のランダム K-fold CV にはどんな落とし穴があるか。


In [ ]:
answers = {
    "1": "",
    "2": "",
    "3": "",
    "4": "",
    "5": "",
    "6": "",
}
answers
